# 2. Evaluatie

In dit hoofdstuk bepalen we met welke metrics kan worden gemeten hoe goed een model presteert op het stroke-voorspellingsprobleem. We demonstreren deze metrics aan de hand van een K-Nearest Neighbors (KNN) model.

## 2.1 Welke metrics zijn relevant?

We voorspellen of iemand een herseninfarct krijgt (ja/nee). Dit is dus een **binaire classificatie**. Bovendien is de dataset **imbalanced**: er zijn veel meer patiënten zonder stroke dan met stroke. Dit heeft grote gevolgen voor welke metrics zinvol zijn.

### Confusion Matrix als basis

Alle metrics komen voort uit vier basiswaarden:

|  | **Voorspeld: Geen stroke** | **Voorspeld: Stroke** |
|---|---|---|
| **Werkelijk: Geen stroke** | True Negative (TN) | False Positive (FP) |
| **Werkelijk: Stroke** | False Negative (FN) ⚠️ | True Positive (TP) |

In een **medische context** is een **False Negative (gemiste stroke) veel erger** dan een False Positive (vals alarm). Een gemiste stroke betekent dat een patiënt met verhoogd risico geen preventieve zorg krijgt.

### De belangrijkste metrics

**Accuracy**
Het percentage patiënten dat correct voorspeld is. Deze metric is **misleidend bij imbalanced data**: een model dat altijd "geen stroke" voorspelt, haalt al rond de 95% accuracy, maar is in de praktijk waardeloos omdat het geen enkele stroke-patiënt herkent.

**Precision**
Van alle patiënten die het model als "stroke" bestempelt, hoeveel hebben er écht een stroke? Een hoge precision betekent weinig vals alarm.

**Recall (Sensitivity)**
Van alle werkelijke stroke-patiënten, hoeveel vangt het model er op? Deze metric is **zeer belangrijk in deze medische context**, omdat we zo min mogelijk patiënten willen missen.

**F1-score**
Het harmonisch gemiddelde van precision en recall. Deze metric is pas hoog als zowel precision als recall hoog zijn. **Dit is de officiële Kaggle-metric** voor deze competitie.

**ROC-AUC**
De oppervlakte onder de ROC-curve. Deze waarde ligt tussen 0.5 (random gokken) en 1.0 (perfect model). Nuttig om modellen onderling te vergelijken, onafhankelijk van de gekozen drempelwaarde.

**Specificity**
Van alle gezonde mensen, hoeveel worden correct als "geen stroke" aangemerkt? Complementair aan recall.

## 2.2 Waarom F1-score de beste maatstaf is voor dit vraagstuk

In de Kaggle-competitie wordt gekozen voor de F1-score om modellen te beoordelen. Dit is om de volgende redenen de meest geschikte maatstaf:

1. **Accuracy faalt bij imbalanced data.** Omdat ongeveer 95% van de patiënten geen stroke heeft, haalt een model dat niemand als stroke-patiënt classificeert al een zeer hoge accuracy, zonder dat het model enige waarde heeft.
2. **Alleen Precision is niet genoeg.** Dit zou een model belonen dat zelden "stroke" voorspelt (hoge zekerheid), maar daardoor veel patiënten mist.
3. **Alleen Recall is ook niet genoeg.** Dit zou een model belonen dat iedereen als stroke bestempelt (geen gemiste patiënten), maar dat veroorzaakt enorm veel vals alarm.
4. **F1-score dwingt een balans af** tussen precision en recall. Een model moet zowel weinig patiënten missen (recall) als nauwkeurig zijn in zijn positieve voorspellingen (precision) om een goede score te halen.
5. In een medische toepassing is dit een redelijke keuze. In een echte productieomgeving zou men mogelijk nog meer gewicht aan recall willen geven (bijvoorbeeld via de F2-score), omdat een gemiste stroke ernstigere gevolgen heeft dan een vals alarm.

## 2.3 Demonstratie met K-Nearest Neighbors

Hieronder trainen we een eenvoudig KNN-model en berekenen we alle bovenstaande metrics om te laten zien hoe deze zich tot elkaar verhouden op dit probleem.


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
y = train["stroke"]
X = train.drop(["stroke", "id"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      
    random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)

print(f"Nauwkeurigheid: {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred))

Nauwkeurigheid: 98.33%
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      6600
           1       0.00      0.00      0.00       110

    accuracy                           0.98      6710
   macro avg       0.49      0.50      0.50      6710
weighted avg       0.97      0.98      0.98      6710



In [21]:
from sklearn.model_selection import cross_val_score

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
y = train["stroke"]
X = train.drop(["stroke", "id"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      
    random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)
scores = cross_val_score(knn, X, y, cv=5) 

#Voor de beste k hebben we een simpele for loop gebruikt en
# Het maakt geen verschil of we k=10 of k=20 gebruiken dus
#hier kiezen we voor k=10
for k in [1, 3, 5, 10, 20]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f"k={k:2d}  :  nauwkeurigheid: {acc:.2%}")



k= 1  :  nauwkeurigheid: 96.86%
k= 3  :  nauwkeurigheid: 98.15%
k= 5  :  nauwkeurigheid: 98.33%
k=10  :  nauwkeurigheid: 98.36%
k=20  :  nauwkeurigheid: 98.36%


Waarom schalen we bij KNN? Afstanden tussen punten zijn bepalend voor welke groep het model kiest om nieuwe datapunten aan toe te voegen, en daarom is het van groot belang dat alle features geschaald worden naar een universele schaal. Anders raakt het model in de war tijdens het fitten van alle grote en kleine waardes door elkaar heen.

Hoe wordt regularisatie toegepast?
Het doel van regularisatie is dat we overfitting tegen willen gaan en dit gebeurt door ten eerste een juiste k te kiezen. Bij een te lage k wordt het model te complex en is heel gevoelig voor uitschieters. Bij een te hoge k wordt het model te algemeen. Het doel is dus om de balans tussen beiden te vinden. Ook wordt er geregulariseerd door gewogen buren. Dit betekent dat buren die dichterbij het nieuwe datapunt liggen zwaarder meewegen. 